# 6 · Rule-based baseline: Linkoln on the "Motivi della decisione" section

This notebook produces the paper appendix **"Rule-based citation extraction as a
baseline"** (Table `tab:linkoln_baseline` and the in-text selection counts 39/39/31).

The paper argues (Section "Design pressures", *Citation networks*) that a citation network
built directly from judgments — even restricting to the reasoning section — includes many
references that are not central to the resolution of the issues, and that issue-level LLM
extraction improves the signal-to-noise ratio. This notebook tests that claim empirically:
it parses all citations from the **"Motivi della decisione"** section with Linkoln (no
LLM involved) and compares them, together with the **full-text** citations and the
**LLM-extracted (post-filter)** citations, against the expert-relevant citation sets.

## Judgment selection

The comparison is meaningful only where the experts' citation lists cover the *whole*
judgment, and where the reasoning section can be located:

1. **Perfect issue recall**: the annotators flagged no missing issues
   (`# questioni presenti NON estratte` = 0 for every annotator who saw the judgment).
   Since the experts compile citation lists per extracted issue, a missed issue would
   leave its references outside the ground truth.
2. **Section detectable**: the regex-based segmenter (`sent_paragraphs_positions_bdgt`,
   from the extraction pipeline) finds the heading "MOTIVI DELLA DECISIONE". The section
   runs from the end of the heading to the start of "P.Q.M." (or the end of the text).

Of the 50 test judgments, 39 satisfy (1), 39 satisfy (2), and **31 satisfy both**.

## Matching

All comparisons in this notebook use **automatic, document-level matching**: every
citation (from the motivi section, the full text, the experts' lists — themselves parsed
with Linkoln — and the LLM output) is reduced to a normalized key (case-law: number +
year; legislation: number + year, completed from the URN or document date when needed;
CELEX code or URN otherwise). References of type *principle* are excluded throughout
(Linkoln does not parse them). Because the matching is automatic, absolute scores are
**not** comparable with the manually matched results of Table `tab:cit_pairwise`; the
comparison *between the three tiers*, which uses identical matching, is consistent.

## Data

`data/linkoln_baseline/` ships the generated inputs: `qualifying_judgments.csv` (selection
flags and section spans), `motivi_citations.csv`, `fulltext_citations.csv` (Linkoln output
on the section / whole text, after the pipeline's standard pre-Linkoln substitutions) and
`expert_citations.csv` (Linkoln output on the experts' citation lists, joint-present
issues only). The generation cell below re-creates them from the judgment texts in
`data/judgments/`, but requires the pipeline utilities (Linkoln jar + Java), which are
not part of this repository; by default it is skipped and the shipped files are used.

In [1]:
import os, re, sys
import numpy as np
import pandas as pd

pd.set_option('display.width', 200)

DATA = '../data'
LB = f'{DATA}/linkoln_baseline'

# Local-only path used by the generation step (not needed for the analysis):
# the extraction-pipeline utilities (segmenter, pre-Linkoln substitutions,
# Linkoln jar + Java server wrapper)
PIPELINE_SRC = ("/Users/k2482382/Library/CloudStorage/OneDrive-King'sCollegeLondon/"
                "code_contenzioso_tributario/src")
REGENERATE = False   # set True to re-run Linkoln (requires Java and the Linkoln jar)

In [2]:
if REGENERATE and os.path.isdir(PIPELINE_SRC):
    import csv as csvmod, io
    sys.path.insert(0, PIPELINE_SRC)
    from utils.sent_paragraphs_position import sent_paragraphs_positions_bdgt
    from utils.preprocessing_linkoln import substitutions_before_linkoln
    from utils.linkoln_server_runner import run_linkoln_string

    LINKOLN_COLS = ("id|source-name|source-partition-id|ref-type|ref-scope|text|context|alias|"
                    "partition|doc-type|authority|region|city|detached-city|section|ministry|"
                    "other-authority|eu-acronym|number|year|full-number|case-number|area|"
                    "doc-date|deposit-date|notification-date|publication-date|applicant|"
                    "defendant|ro-number|rv-number|gu-series|url|cited-doc-simple-id|"
                    "cited-art-simple-id|cited-comma-simple-id").split("|")

    def parse_server_line(line):
        """Linkoln's CSV quotes fields containing commas; the LinkolnServer wrapper
        replaces every comma with '|', so we re-split on '|' respecting the quotes
        and restore the commas inside quoted fields."""
        vals = next(csvmod.reader(io.StringIO(line), delimiter='|', quotechar='"'))
        vals = [v.replace('|', ',') for v in vals]
        vals += [''] * (len(LINKOLN_COLS) - len(vals))
        return vals[:len(LINKOLN_COLS)]

    def linkoln_rows(text, extra):
        out = run_linkoln_string(substitutions_before_linkoln(text, verbose=0))
        return [{**extra, **dict(zip(LINKOLN_COLS, parse_server_line(l)))}
                for l in out.split('\n') if l.strip() and not l.startswith('ERROR')]

    COL_PRESENT = 'questione presente nella sentenza (TRUE/FALSE)'
    da = pd.read_csv(f'{DATA}/validation_annotator_A1.csv')
    dp = pd.read_csv(f'{DATA}/validation_annotator_A2.csv')
    NE = [c for c in da.columns if c.startswith('# questioni presenti NON')][0]
    sa, sp = set(da['Sentenza']), set(dp['Sentenza'])
    common = sa & sp

    def perfect_recall(d):
        g = d.groupby('Sentenza')[NE].max().fillna(0)
        return set(g[g == 0].index)
    pr_a, pr_p = perfect_recall(da), perfect_recall(dp)
    pr = {s for s in sa | sp if (s not in sa or s in pr_a) and (s not in sp or s in pr_p)}

    cat = pd.concat([da.assign(annotator='A1'), dp.assign(annotator='A2')], ignore_index=True)
    sh = cat[cat['Sentenza'].isin(common)].copy()
    sh[COL_PRESENT] = sh[COL_PRESENT].astype(bool)
    joint = sh.groupby(['Sentenza', 'Questioni estratte'])[COL_PRESENT].all()
    def jp(d):
        o = d.copy(); o['j'] = o[COL_PRESENT].astype(bool)
        m = o['Sentenza'].isin(common)
        keys = pd.MultiIndex.from_arrays([o.loc[m, 'Sentenza'], o.loc[m, 'Questioni estratte']])
        o.loc[m, 'j'] = joint.reindex(keys).values
        return o
    da, dp = jp(da), jp(dp)

    rows, motivi_rows, expert_rows, fulltext_rows = [], [], [], []
    for s in sorted(sa | sp):
        stem = s.replace('.json', '')
        txt = open(f'{DATA}/judgments/{stem}.txt').read()
        pos = sent_paragraphs_positions_bdgt(txt)
        m, q = pos['motivi della decisione'], pos['p.q.m.']
        sec_start = m[1] if m else None
        sec_end = q[0] if (m and q and q[0] > m[1]) else (len(txt) if m else None)
        qualifies = (s in pr) and (m is not None)
        rows.append({'filename': s, 'perfect_issue_recall': s in pr, 'motivi_found': m is not None,
                     'section_start': sec_start, 'section_end': sec_end, 'qualifies': qualifies})
        if not qualifies:
            continue
        motivi_rows += linkoln_rows(txt[sec_start:sec_end], {'filename': s})
        fulltext_rows += linkoln_rows(txt, {'filename': s})
        for tag, d in [('A1', da), ('A2', dp)]:
            sub = d[(d['Sentenza'] == s) & (d['j'] == True)]
            for _, r in sub.iterrows():
                cell = r['Lista citazioni esperto']
                if pd.isna(cell) or str(cell).strip() in ('', '/'):
                    continue
                expert_rows += linkoln_rows(str(cell), {'filename': s, 'annotator': tag,
                                                        'questione': r['Questioni estratte'][:60]})
        print(s, 'done')

    pd.DataFrame(rows).to_csv(f'{LB}/qualifying_judgments.csv', index=False)
    pd.DataFrame(motivi_rows).to_csv(f'{LB}/motivi_citations.csv', sep='|', index=False)
    pd.DataFrame(expert_rows).to_csv(f'{LB}/expert_citations.csv', sep='|', index=False)
    pd.DataFrame(fulltext_rows).to_csv(f'{LB}/fulltext_citations.csv', sep='|', index=False)
else:
    print('Generation skipped — using the CSVs shipped in data/linkoln_baseline/.')

Generation skipped — using the CSVs shipped in data/linkoln_baseline/.


## Citation keys and the three citation tiers

In [3]:
def _c(x):
    s = str(x).strip().strip('"').strip()
    return '' if s.lower() in ('', 'nan', 'none') else s


def ref_key(row):
    """Normalized document-level key for a Linkoln-parsed citation."""
    rt   = _c(row.get('ref-type'))
    num  = _c(row.get('number'))
    year = _c(row.get('year'))
    if not year:                                  # Linkoln often returns the year in a date field
        for dc in ('doc-date', 'deposit-date'):
            m = re.match(r'(\d{4})', _c(row.get(dc)))
            if m:
                year = m.group(1); break
    url   = _c(row.get('url'))
    alias = _c(row.get('cited-doc-simple-id')) or _c(row.get('alias'))
    if not (num and year) and 'urn:nir' in url:   # complete number/year from the URN
        m = re.search(r'urn:nir:[^~&]*?(\d{4})(?:-\d\d-\d\d)?;([\w\-.]+)', url)
        if m:
            year = year or m.group(1)
            num = num or re.sub(r'\D', '', m.group(2))
    if num and year:
        try:
            return ('cl' if rt == 'caselaw' else 'leg', int(float(num)), int(float(year)))
        except ValueError:
            pass
    if 'celex' in url.lower():
        m = re.search(r'(?i)celex[:%3A]*([0-9A-Za-z()]+)', url)
        if m:
            return ('celex', m.group(1))
    m = re.search(r'urn:nir:([^~&]+)', url)
    if m:
        return ('urn', m.group(1))
    if alias:
        return ('doc', alias)
    return None


def keyset(g):
    return {k for k in (ref_key(r) for _, r in g.iterrows()) if k}


qual = pd.read_csv(f'{LB}/qualifying_judgments.csv')
Q = set(qual[qual['qualifies']]['filename'])
mot = pd.read_csv(f'{LB}/motivi_citations.csv', sep='|')
ful = pd.read_csv(f'{LB}/fulltext_citations.csv', sep='|')
exp = pd.read_csv(f'{LB}/expert_citations.csv', sep='|')

mot_keys = {s: keyset(g) for s, g in mot.groupby('filename')}
ful_keys = {s: keyset(g) for s, g in ful.groupby('filename')}
exp_keys = {(s, a): keyset(g) for (s, a), g in exp.groupby(['filename', 'annotator'])}
print(f"qualifying judgments: {len(Q)}")
print(f"unkeyed rows — motivi: {sum(ref_key(r) is None for _, r in mot.iterrows())}, "
      f"full text: {sum(ref_key(r) is None for _, r in ful.iterrows())}, "
      f"expert: {sum(ref_key(r) is None for _, r in exp.iterrows())}")

qualifying judgments: 31
unkeyed rows — motivi: 12, full text: 17, expert: 2


In [4]:
# LLM tier: post-filter (kept) references of joint-present issues, principles excluded
# (filter decisions recomputed exactly as in notebook 05)
sys.path.append('../src')
from principi_classifier import classify

df = pd.read_csv(f'{DATA}/hallucination/references_test_set.csv', sep='|')
df = df[df['questione_element'] == 'lista_riferimenti_diritto'].copy()
df['row_verified'] = ~df['hallucinated'].isin([np.nan, '', 'True'])
items = (df.groupby(['filename', 'id_questione', 'reference_id'])
           .agg(reference_type=('reference_type', 'first'),
                original_text=('original_text', 'first'),
                text_match=('row_verified', 'any')).reset_index())
pr = items['reference_type'] == 'princ'
items['kept'] = items['text_match']
items.loc[pr, 'kept'] = items.loc[pr].apply(
    lambda r: bool(r['text_match'] and classify(r['original_text']).exists), axis=1)
HALLUCINATED_ISSUES = {('Sentenza_V43_3256_2021.xml', 'Q1'), ('Sentenza_Z29_1812_2023.xml', 'Q2'),
                       ('Sentenza_V16_372_2022.xml', 'Q1'), ('Sentenza_V22_5547_2023.xml', 'Q2')}
items = items[~items.apply(lambda r: (r['filename'], r['id_questione']) in HALLUCINATED_ISSUES, axis=1)]
kept = set(map(tuple, items[items['kept']][['filename', 'id_questione', 'reference_id']].values))
df['item_key'] = list(map(tuple, df[['filename', 'id_questione', 'reference_id']].values))
df_kept = df[(df['item_key'].isin(kept)) & (df['reference_type'] != 'princ')]
llm_keys = {s.replace('.xml', '.json'): keyset(g) for s, g in df_kept.groupby('filename')
            if s.replace('.xml', '.json') in Q}

## Precision and recall of the three tiers against the expert-relevant citations

For each annotator, pooled over the qualifying judgments they annotated (P = fraction of
the tier's citations deemed relevant by the expert; R = fraction of the expert's citations
recovered). 95% CIs from the judgment-level cluster bootstrap (B=10,000).

In [5]:
def tier_table(tag):
    judgs = sorted({s for (s, a) in exp_keys if a == tag and s in Q})
    per_j = []
    for s in judgs:
        e = exp_keys[(s, tag)]
        row = {'judgment': s, 'n_E': len(e)}
        for name, ks in [('full', ful_keys), ('motivi', mot_keys), ('llm', llm_keys)]:
            t = ks.get(s, set())
            row[f'n_{name}'] = len(t)
            row[f'n_{name}_E'] = len(t & e)
        per_j.append(row)
    dj = pd.DataFrame(per_j)

    rng = np.random.default_rng(0)
    idx = rng.integers(0, len(dj), size=(10_000, len(dj)))
    out = []
    for name, label in [('full', 'Full judgment (Linkoln)'),
                        ('motivi', 'Motivi section (Linkoln)'),
                        ('llm', 'LLM, post-filter')]:
        T, E, TE = dj[f'n_{name}'].sum(), dj['n_E'].sum(), dj[f'n_{name}_E'].sum()
        Tb = dj[f'n_{name}'].to_numpy(float)[idx].sum(axis=1)
        Eb = dj['n_E'].to_numpy(float)[idx].sum(axis=1)
        TEb = dj[f'n_{name}_E'].to_numpy(float)[idx].sum(axis=1)
        with np.errstate(divide='ignore', invalid='ignore'):
            Pb, Rb = TEb / Tb, TEb / Eb
        plo, phi = np.nanpercentile(Pb, [2.5, 97.5])
        rlo, rhi = np.nanpercentile(Rb, [2.5, 97.5])
        out.append({'Tier': label,
                    'Citations/judgment': f'{dj[f"n_{name}"].mean():.2f}',
                    'P': f'{TE/T*100:.1f} [{plo*100:.1f}, {phi*100:.1f}]',
                    'R': f'{TE/E*100:.1f} [{rlo*100:.1f}, {rhi*100:.1f}]'})
    print(f"vs {tag} — {len(dj)} qualifying judgments, "
          f"{dj['n_E'].sum()} expert-relevant citations ({dj['n_E'].mean():.2f}/judgment)")
    return pd.DataFrame(out).set_index('Tier')

for tag in ['A1', 'A2']:
    print(tier_table(tag).to_string(), end='\n\n')

vs A1 — 19 qualifying judgments, 77 expert-relevant citations (4.05/judgment)
                         Citations/judgment                  P                   R
Tier                                                                              
Full judgment (Linkoln)                6.74  57.8 [43.8, 69.3]  96.1 [88.9, 100.0]
Motivi section (Linkoln)               4.79  81.3 [70.4, 90.7]  96.1 [88.9, 100.0]
LLM, post-filter                       3.53  88.1 [73.8, 97.9]   76.6 [68.1, 90.3]

vs A2 — 20 qualifying judgments, 68 expert-relevant citations (3.40/judgment)
                         Citations/judgment                  P                  R
Tier                                                                             
Full judgment (Linkoln)                7.60  39.5 [26.9, 53.5]  88.2 [77.8, 98.4]
Motivi section (Linkoln)               5.60  53.6 [33.8, 80.0]  88.2 [77.8, 98.4]
LLM, post-filter                       3.55  74.6 [54.0, 91.0]  77.9 [66.7, 89.7]



In [6]:
# Sanity check: post-filter LLM citations should mostly lie inside the motivi section
in_mot = tot = 0
for s, l in llm_keys.items():
    tot += len(l)
    in_mot += len(l & mot_keys.get(s, set()))
print(f"LLM post-filter citations found in the motivi section: {in_mot}/{tot} ({in_mot/tot:.0%})")
print("(the rest are cited elsewhere in the judgment or normalize to slightly different keys)")

LLM post-filter citations found in the motivi section: 101/114 (89%)
(the rest are cited elsewhere in the judgment or normalize to slightly different keys)


## Reading the result

Both rule-based tiers over-generate relative to what the experts consider relevant to the
resolution of the issues, and the dilution grows with the extraction scope: the full
judgment contains the most citations and the lowest precision, the motivi section sits in
between, and the LLM tier is the most precise while conceding only a moderate amount of
recall. Restricting Linkoln to the reasoning section is therefore a real but partial
remedy: a substantial share of the section's citations are background, party-argument or
boilerplate references that the experts do not regard as load-bearing, which is precisely
the signal-to-noise argument made in the paper's *Citation networks* paragraph — with the
caveats that the sample is 31 judgments and the matching is automatic rather than
expert-validated.